# Question Analysis

In [ ]:
# --- Private study data: NOT in this repository (see README.md) ---
# Point STUDY_DIR at your copy of the archived LAK 2026 study artifacts.
from pathlib import Path

STUDY_DIR = Path(".")
SURVEY_QUESTIONS = STUDY_DIR / "results/qualtrics/AP Environment/survey_questions.jsonl"
QUALTRICS_EXPORT = STUDY_DIR / "qualtrics/New-AP-Environment_February 21, 2025_10.55.csv"
GEMINI_RUN_DIR = STUDY_DIR / "results/gemini/Sat_Sep_27_17-04-30_2025"

FIGURES_DIR = Path("figures")  # regenerated locally; git-ignored (run outputs are never committed)
FIGURES_DIR.mkdir(exist_ok=True)

In [ ]:
from sklearn.metrics import cohen_kappa_score
import itertools
import numpy as np
import pandas as pd

def create_kappa_matrix(df_ratings):
    # Get the list of raters (column names)
    raters = df_ratings.columns

    # Initialize a square DataFrame to store the kappa values
    kappa_matrix = pd.DataFrame(np.nan, index=raters, columns=raters)
    
    # Use itertools.combinations to get all unique pairs of raters
    for rater1, rater2 in itertools.combinations(raters, 2):
        # Get the ratings for the two raters
        ratings1 = df_ratings[rater1]
        ratings2 = df_ratings[rater2]
        
        # Calculate Cohen's Kappa
        kappa = cohen_kappa_score(ratings1, ratings2)
        
        # Store the kappa value in the matrix (it's symmetric)
        kappa_matrix.loc[rater1, rater2] = kappa
        kappa_matrix.loc[rater2, rater1] = kappa
        
    # Fill the diagonal with 1s, as a rater always agrees perfectly with themselves
    np.fill_diagonal(kappa_matrix.values, 1.0)
    
    return kappa_matrix

In [ ]:
from kcluster.io.jsonl import load_questions

questions = load_questions(str(SURVEY_QUESTIONS))
len(questions)

In [ ]:
choices = dict()
answers = dict()
aligned = dict()

for q in questions:
    choices[q["id"]] = {item["text"]: item["label"] for item in q["question"]["choices"]}
    choices[q["id"]].update({"None of the above": "e"})
    answers[q["id"]] = q.answer
    aligned[q["id"]] = "No" if "false_lo" in q else "Yes"

In [ ]:
# Load human response data (Qualtrics export; response columns only — the
# export's IP/geo/email columns are never read)
import pandas as pd

df = pd.read_csv(QUALTRICS_EXPORT)
res_df = df.filter(regex=r"^[a-z0-9]+(\-LO)?$").iloc[2:].reset_index(drop=True)
res_df

## Q1: Do teachers' answer align with Phi-2's?

In [ ]:
# Collect human responses for Q1
q1_df = res_df.iloc[:, ::2].copy()
for col in q1_df:
    q1_df[col] = q1_df[col].apply(lambda x: choices[col][x])
q1_df = q1_df.T
q1_df = q1_df.rename(columns=lambda x: f"P{x+1}")
q1_df

In [ ]:
# Phi-2 answers
phi2_q1_answers = pd.Series(answers, name="Phi-2")
phi2_q1_answers

In [ ]:
# Gemini answers (from papers/lak2026/scripts/gemini_logprob_judge.py)
gemini_q1_df = pd.read_csv(GEMINI_RUN_DIR / "q1-answers.csv")
gemini_q1_df = gemini_q1_df.set_index("id").rename(columns={"answer": "Gemini"})
gemini_q1_answers = gemini_q1_df.squeeze()
gemini_q1_answers

In [ ]:
all_q1_answers = q1_df.join(phi2_q1_answers).join(gemini_q1_answers)
all_q1_answers

### Pairwise Cohen's Kappa

In [ ]:
kappa_matrix = create_kappa_matrix(all_q1_answers)
kappa_matrix

In [ ]:
kappa_values = dict()

inds = np.triu_indices(q1_df.shape[1], k=1)
kappa_values["Human-Human"] = np.mean(kappa_matrix.values[inds])

kappa_values["Human-Phi2"] = kappa_matrix.loc["Phi-2", q1_df.columns].mean()
kappa_values["Human-Gemini"] = kappa_matrix.loc["Gemini", q1_df.columns].mean()
kappa_values["Phi2-Gemini"] = kappa_matrix.loc["Phi-2", "Gemini"]
kappa_values

### Bar chart of average kappa scores

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Set the font to a serif font similar to Linux Libertine
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Linux Libertine"],  # You can specify a list of serif fonts, e.g., ["Linux Libertine", "Times New Roman"]
})


# compute standard errors for each metric (Phi2-Gemini will have 0)
n_hum = q1_df.shape[1]
human_cols = list(q1_df.columns)

# Human-Human: use upper-triangle pairwise human-human kappas
hh_vals = kappa_matrix.loc[human_cols, human_cols].values
hh_pairs = hh_vals[np.triu_indices(n_hum, k=1)]
sem_hh = hh_pairs.std(ddof=1) / np.sqrt(len(hh_pairs))

# Human-Phi2 and Human-Gemini: variability across humans
sem_phi2 = kappa_matrix.loc["Phi-2", human_cols].std(ddof=1) / np.sqrt(n_hum)
sem_gemini = kappa_matrix.loc["Gemini", human_cols].std(ddof=1) / np.sqrt(n_hum)

sems = {"Human-Human": sem_hh, "Human-Phi2": sem_phi2, "Human-Gemini": sem_gemini, "Phi2-Gemini": 0.0}

# Convert the dictionary to a pandas Series for easier plotting
kappa_series = pd.Series(kappa_values)#.sort_values(ascending=True)
heights = kappa_series.values
labels = kappa_series.index.tolist()
errors = np.array([sems[l] for l in labels])

# Set a professional plot style
plt.style.use('seaborn-v0_8-whitegrid')

# Create the plot with specified width and DPI for print quality
fig, ax = plt.subplots(figsize=(4, 3.5), dpi=300)
ax.grid(False, axis='x')  # Disable vertical grid lines

# Colors
colors = sns.color_palette("viridis", len(labels))

# Bar plot with error bars (Phi2-Gemini has zero error)
x = np.arange(len(labels))
bars = ax.bar(x, heights, width=0.4, color=colors, yerr=errors, capsize=4, edgecolor='none')

# Add a horizontal line for the Human-Human baseline
baseline_value = kappa_values["Human-Human"]
ax.axhline(y=baseline_value, color='r', linestyle='--', label=f'Human-Human Baseline ({baseline_value:.2f})')

# Add data labels on top of each bar (place above error bar if present)
for i, bar in enumerate(bars):
    h = bar.get_height()
    err = errors[i]
    y = h + (err if err > 0 else 0) + 0.02
    ax.annotate(f"{h:.2f}", 
                (bar.get_x() + bar.get_width() / 2., y), 
                ha='center', va='bottom',
                fontsize=8)

# Customize plot labels and title
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8, rotation=0)
ax.set_title(r"Average Pairwise Inter-Rater Agreement (Cohen's Kappa)", fontsize=9, pad=10)
ax.set_ylabel(r"Cohen's Kappa Score", fontsize=9)
ax.set_xlabel("")
ax.set_ylim(0, 1)
ax.tick_params(axis='y', labelsize=9)
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
sems

In [ ]:
fig.savefig(FIGURES_DIR / "q1-avg_cohen_kappa.pdf", dpi=300, bbox_inches='tight')

In [ ]:
# import seaborn as sns
# import pandas as pd
# import matplotlib.pyplot as plt

# # Set the font to a serif font similar to Linux Libertine
# plt.rcParams.update({
#     "font.family": "serif",
#     "font.serif": ["Linux Libertine"],  # You can specify a list of serif fonts, e.g., ["Linux Libertine", "Times New Roman"]
# })


# # Convert the dictionary to a pandas Series for easier plotting
# kappa_series = pd.Series(kappa_values).sort_values(ascending=True)

# # Set a professional plot style
# plt.style.use('seaborn-v0_8-whitegrid')

# # Create the plot with specified width and DPI for print quality
# fig, ax = plt.subplots(figsize=(4, 3.5), dpi=300)

# # Create the bar plot with thinner bars
# sns.barplot(x=kappa_series.index, y=kappa_series.values, ax=ax, palette="viridis", width=0.4)

# # Add a horizontal line for the Human-Human baseline
# baseline_value = kappa_values["Human-Human"]
# ax.axhline(y=baseline_value, color='r', linestyle='--', label=f'Human-Human Baseline ({baseline_value:.2f})')

# # Add data labels on top of each bar
# for p in ax.patches:
#     ax.annotate(format(p.get_height(), '.2f'), 
#                    (p.get_x() + p.get_width() / 2., p.get_height()), 
#                    ha = 'center', va = 'center', 
#                    xytext = (0, 4.5), 
#                    textcoords = 'offset points',
#                    fontsize=8)

# # Customize plot labels and title
# ax.set_title(r"Average Pairwise Inter-Rater Agreement (Cohen's Kappa)", fontsize=9, pad=10,)
# ax.set_ylabel(r"Cohen's Kappa Score", fontsize=9)
# ax.set_xlabel("")
# ax.set_ylim(0, 1) # Kappa scores range from -1 to 1, but agreement is typically 0-1
# ax.tick_params(axis='x', labelsize=8, rotation=0)
# ax.tick_params(axis='y', labelsize=9)
# ax.legend(fontsize=8)

# # Ensure the plot is displayed cleanly
# plt.tight_layout()
# plt.show()

In [ ]:
# import seaborn as sns
# import pandas as pd
# import matplotlib.pyplot as plt

# # Convert the dictionary to a pandas Series for easier plotting
# kappa_series = pd.Series(kappa_values).sort_values(ascending=True)

# # Set a professional plot style
# plt.style.use('seaborn-v0_8-whitegrid')

# # Create the plot with specified width and DPI for print quality
# fig, ax = plt.subplots(figsize=(4.5, 3), dpi=300)

# # Create the bar plot with thinner bars
# sns.barplot(x=kappa_series.index, y=kappa_series.values, ax=ax, palette="viridis", width=0.4)

# # Add a horizontal line for the Human-Human baseline
# baseline_value = kappa_values["Human-Human"]
# ax.axhline(y=baseline_value, color='r', linestyle='--', label=f'Human-Human Baseline ({baseline_value:.2f})')

# # Add data labels on top of each bar
# for p in ax.patches:
#     ax.annotate(format(p.get_height(), '.2f'), 
#                    (p.get_x() + p.get_width() / 2., p.get_height()), 
#                    ha = 'center', va = 'center', 
#                    xytext = (0, 5), 
#                    textcoords = 'offset points',
#                    fontsize=8)

# # Customize plot labels and title
# ax.set_title(r"Average Pairwise Inter-Rater Agreement (Cohen's $\kappa$)", fontsize=9, pad=10,)
# ax.set_ylabel(r"Cohen's Kappa Score", fontsize=9)
# ax.set_xlabel("")
# ax.set_ylim(0, 1) # Kappa scores range from -1 to 1, but agreement is typically 0-1
# ax.tick_params(axis='x', labelsize=8, rotation=0)
# ax.tick_params(axis='y', labelsize=9)
# ax.legend(fontsize=8)

# # Ensure the plot is displayed cleanly
# plt.tight_layout()
# plt.show()

### Heatmap of Cohen's Kappa Scores

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


# Create a mask for the upper triangle to make the plot cleaner (it's symmetric)
mask = np.tril(np.ones_like(kappa_matrix, dtype=bool), k=0)

# Set a professional plot style
plt.style.use('seaborn-v0_8-whitegrid')
# Remove grid lines from the plot style
ax.grid(False)

fig, ax = plt.subplots(figsize=(4, 3.6), dpi=300)

# Create the heatmap with several enhancements
sns.heatmap(
    kappa_matrix,
    mask=mask,          # Apply the mask
    annot=True,         # Display the kappa values in the cells
    fmt=".2f",          # Format the numbers to two decimal places
    cmap='viridis',     # Use a colorblind-friendly and professional palette
    linewidths=0,      # Remove lines between cells
    ax=ax,
    annot_kws={"size": 6},
    # cbar_kws={'label': "Cohen's Kappa Score"} # Add a label to the color bar
)

# Fill the diagonal with a gray color to indicate self-agreement is 1.0 by definition
for i in range(kappa_matrix.shape[0]):
    ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=True, color='lightgray', lw=0))
    ax.text(i + 0.5, i + 0.5, "1.0", ha='center', va='center', color='black', fontsize=6)#, fontweight='bold')

# Invert the y-axis to have the matrix origin at the top-left
ax.invert_yaxis()

# Set plot title and labels with appropriate font sizes
ax.set_title("All Pairwise Inter-Rater Agreement (Cohen's Kappa)", fontsize=9, pad=10)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0, labelsize=8)
ax.figure.axes[-1].yaxis.label.set_size(8)

# Ensure the plot is displayed cleanly
plt.tight_layout()

plt.show()

In [ ]:
fig.savefig(FIGURES_DIR / "q1-heatmap_cohen_kappa.pdf", dpi=300, bbox_inches='tight')

### Fleiss' Kappa

In [ ]:
from statsmodels.stats.inter_rater import aggregate_raters, fleiss_kappa

fleiss_values = dict()
# Human-Human Fleiss' Kappa
human_raters = q1_df.columns.tolist()
data, n_cat = aggregate_raters(all_q1_answers[human_raters])
fleiss_values["Human"] = fleiss_kappa(data, method='fleiss')

data, n_cat = aggregate_raters(all_q1_answers[human_raters + ["Phi-2"]])
fleiss_values["Human + Phi-2"] = fleiss_kappa(data, method='fleiss')

data, n_cat = aggregate_raters(all_q1_answers[human_raters + ["Gemini"]])
fleiss_values["Human + Gemini"] = fleiss_kappa(data, method='fleiss')

data, n_cat = aggregate_raters(all_q1_answers[human_raters + ["Phi-2", "Gemini"]])
fleiss_values["Human + Both"] = fleiss_kappa(data, method='fleiss')

fleiss_values

#### Annotation Matrix

In [ ]:
def create_annotation_matrix_plot(df_ratings):
    """
    Visualizes raw categorical ratings in a matrix where each cell is colored
    by its answer choice.

    Args:
        df_ratings (pd.DataFrame): DataFrame where rows are items (MCQs) and
                                   columns are raters.
    """
    # --- 1. Prepare Data for Plotting ---
    
    # We need to map the categorical choices ('a', 'b', etc.) to numbers
    # so seaborn can assign them unique colors.
    choices = ["a", "b", "c", "d", "e"]
    choice_to_num = {choice: i for i, choice in enumerate(choices)}
    
    # Create a numerical version of the dataframe
    numerical_df = df_ratings.apply(lambda s: s.map(choice_to_num))
    
    # --- 2. Create the Visualization ---
    
    plt.style.use('seaborn-v0_8-whitegrid')
    # The figure needs to be wide to accommodate 64 questions
    fig, ax = plt.subplots(figsize=(7, 4), dpi=300)
    
    # Use a qualitative colormap with enough distinct colors
    cmap = plt.cm.get_cmap('tab10', len(choices))

    sns.heatmap(
        numerical_df.T,  # Transpose the matrix so raters are on the y-axis
        annot=False,     # Annotations would make this too cluttered
        cmap=cmap,
        linewidths=.5,
        linecolor='lightgray',
        cbar=True,       # We will use a color bar as a legend
        ax=ax
    )

    # --- 3. Format the Plot for Clarity ---

    # Customize the color bar to act as a legend for the answer choices
    colorbar = ax.collections[0].colorbar
    # Set the tick locations to the center of each color segment
    tick_locs = np.arange(len(choices)) + 0.5
    colorbar.set_ticks(tick_locs)
    # Set the tick labels to the actual answer choices
    colorbar.set_ticklabels(choices)
    colorbar.set_label('Answer Choice', fontsize=12)

    # Format the axes
    ax.set_title('Per-Question Answer Choices by Rater', fontsize=20, pad=20)
    ax.set_xlabel('Question Number', fontsize=14)
    ax.set_ylabel('Rater', fontsize=14)
    
    # Make x-axis labels readable by showing only every 5th question number
    question_labels = [str(i + 1) if (i + 1) % 5 == 0 else '' for i in range(len(df_ratings.index))]
    ax.set_xticks(np.arange(len(df_ratings.index)) + 0.5)
    ax.set_xticklabels(question_labels, rotation=0)

    plt.tight_layout()
    plt.show()

In [ ]:
create_annotation_matrix_plot(all_q1_answers)

### Majority voting

In [ ]:
all_q1_answers[q1_df.eq("e").sum(axis=1).eq(1)]

In [ ]:
all_q1_answers.loc["71da485b88f92bd7a4a1a38b6c5b03d2"]

In [ ]:
q1_majority = q1_df.mode(axis=1).dropna(axis=1).rename(columns={0: "Human Majority"}).squeeze()
q1_majority

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
# Set the font to a serif font similar to Linux Libertine
plt.rcParams.update({

    "font.family": "serif",
    "font.serif": ["Linux Libertine"],  # You can specify a list of serif fonts, e.g., ["Linux Libertine", "Times New Roman"]
})
# Set a professional plot style
plt.style.use('seaborn-v0_8-whitegrid')
# Define true and predicted labels
y_true = q1_majority
y_pred = all_q1_answers["Phi-2"]

confusion_matrix = confusion_matrix(y_true, y_pred, labels=["a", "b", "c", "d", "e"])
confusion_matrix

In [ ]:
# human majority vs. Phi-2
cohen_kappa_score(q1_majority, phi2_q1_answers)

In [ ]:
cohen_kappa_score(q1_majority, gemini_q1_answers)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(q1_majority, phi2_q1_answers, labels=["a", "b", "c", "d", "e"]))

In [ ]:
q1_majority[~q1_majority.eq(phi2_q1_answers)]

In [ ]:
# human majority vs. Gemini
q1_majority[~q1_majority.eq(gemini_q1_answers)]

In [ ]:
inds = q1_majority[~q1_majority.eq(gemini_q1_answers)].index.intersection(
    q1_majority[~q1_majority.eq(phi2_q1_answers)].index
)
inds

In [ ]:
all_q1_answers.loc[inds]

In [ ]:
for q in questions:
    if q["id"] in inds:
        print(str(q))

In [ ]:
from sklearn.metrics import cohen_kappa_score

cohen_kappa_score(q1_majority.squeeze(), phi2_q1_ser)

In [ ]:
cohen_kappa_score(gemini_q1_df, phi2_q1_ser)

## Q2: Can teachers identify aligned and misaligned learning objectives?

In [ ]:
res_df

In [ ]:
# Collect human responses for Q2
q2_df = res_df.iloc[:, 1::2].copy().rename(columns=lambda x: x.rstrip("-LO")).T
q2_df = q2_df.rename(columns=lambda x: f"P{x+1}")
q2_df

In [ ]:
# Collect Phi-2 responses
phi2_q2_answers = pd.Series(aligned, name="Phi-2")
phi2_q2_answers

In [ ]:
gemini_q2_df = pd.read_csv(GEMINI_RUN_DIR / "q2-answers.csv")
gemini_q2_df = gemini_q2_df.set_index("id").rename(columns={"answer": "Gemini"})
gemini_q2_df["Gemini"] = gemini_q2_df["Gemini"].str.capitalize()
gemini_q2_answers = gemini_q2_df.squeeze()
gemini_q2_answers

In [ ]:
all_q2_answers = q2_df.join(phi2_q2_answers).join(gemini_q2_answers)
all_q2_answers

### Pairwise Cohen's Kappa

In [ ]:
kappa_matrix = create_kappa_matrix(all_q2_answers)
kappa_matrix

In [ ]:
kappa_values = dict()

inds = np.triu_indices(q2_df.shape[1], k=1)
kappa_values["Human-Human"] = np.mean(kappa_matrix.values[inds])

kappa_values["Human-Phi2"] = kappa_matrix.loc["Phi-2", q2_df.columns].mean()
kappa_values["Human-Gemini"] = kappa_matrix.loc["Gemini", q2_df.columns].mean()
kappa_values["Phi2-Gemini"] = kappa_matrix.loc["Phi-2", "Gemini"]
kappa_values

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Set the font to a serif font similar to Linux Libertine
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Linux Libertine"],  # You can specify a list of serif fonts, e.g., ["Linux Libertine", "Times New Roman"]
})

# compute standard errors for each metric (Phi2-Gemini will have 0)
n_hum = q2_df.shape[1]
human_cols = list(q2_df.columns)

# Human-Human: use upper-triangle pairwise human-human kappas
hh_vals = kappa_matrix.loc[human_cols, human_cols].values
hh_pairs = hh_vals[np.triu_indices(n_hum, k=1)]
sem_hh = hh_pairs.std(ddof=1) / np.sqrt(len(hh_pairs))

# Human-Phi2 and Human-Gemini: variability across humans
sem_phi2 = kappa_matrix.loc["Phi-2", human_cols].std(ddof=1) / np.sqrt(n_hum)
sem_gemini = kappa_matrix.loc["Gemini", human_cols].std(ddof=1) / np.sqrt(n_hum)

sems = {"Human-Human": sem_hh, "Human-Phi2": sem_phi2, "Human-Gemini": sem_gemini, "Phi2-Gemini": 0.0}

# Convert the dictionary to a pandas Series for easier plotting
kappa_series = pd.Series(kappa_values)#.sort_values(ascending=True)
heights = kappa_series.values
labels = kappa_series.index.tolist()
errors = np.array([sems[l] for l in labels])

# Set a professional plot style
plt.style.use('seaborn-v0_8-whitegrid')

# Create the plot with specified width and DPI for print quality
fig, ax = plt.subplots(figsize=(4, 3.5), dpi=300)
ax.grid(False, axis='x')  # Disable vertical grid lines

# Colors
colors = sns.color_palette("viridis", len(labels))

# Bar plot with error bars (Phi2-Gemini has zero error)
x = np.arange(len(labels))
bars = ax.bar(x, heights, width=0.4, color=colors, yerr=errors, capsize=4, edgecolor='none')

# Add a horizontal line for the Human-Human baseline
baseline_value = kappa_values["Human-Human"]
ax.axhline(y=baseline_value, color='r', linestyle='--', label=f'Human-Human Baseline ({baseline_value:.2f})')

# Add data labels on top of each bar (place above error bar if present)
for i, bar in enumerate(bars):
    h = bar.get_height()
    err = errors[i]
    y = h + (err if err > 0 else 0) + 0.02
    ax.annotate(f"{h:.2f}", 
                (bar.get_x() + bar.get_width() / 2., y), 
                ha='center', va='bottom',
                fontsize=8)

# Customize plot labels and title
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8, rotation=0)
ax.set_title(r"Average Pairwise Inter-Rater Agreement (Cohen's Kappa)", fontsize=9, pad=10)
ax.set_ylabel(r"Cohen's Kappa Score", fontsize=9)
ax.set_xlabel("")
ax.set_ylim(0, 1)
ax.tick_params(axis='y', labelsize=9)
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
sems

In [ ]:
fig.savefig(FIGURES_DIR / "q2-avg_cohen_kappa.pdf", dpi=300, bbox_inches='tight')

### Heatmap of Cohen's Kappa Scores

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


# Create a mask for the upper triangle to make the plot cleaner (it's symmetric)
mask = np.tril(np.ones_like(kappa_matrix, dtype=bool), k=0)

# Set a professional plot style
plt.style.use('seaborn-v0_8-whitegrid')
# Remove grid lines from the plot style
ax.grid(False)

fig, ax = plt.subplots(figsize=(4, 3.6), dpi=300)

# Create the heatmap with several enhancements
sns.heatmap(
    kappa_matrix,
    mask=mask,          # Apply the mask
    annot=True,         # Display the kappa values in the cells
    fmt=".2f",          # Format the numbers to two decimal places
    cmap='viridis',     # Use a colorblind-friendly and professional palette
    linewidths=0,      # Remove lines between cells
    ax=ax,
    annot_kws={"size": 6},
    # cbar_kws={'label': "Cohen's Kappa Score"} # Add a label to the color bar
)

# Fill the diagonal with a gray color to indicate self-agreement is 1.0 by definition
for i in range(kappa_matrix.shape[0]):
    ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=True, color='lightgray', lw=0))
    ax.text(i + 0.5, i + 0.5, "1.0", ha='center', va='center', color='black', fontsize=6)#, fontweight='bold')

# Invert the y-axis to have the matrix origin at the top-left
ax.invert_yaxis()

# Set plot title and labels with appropriate font sizes
ax.set_title("All Pairwise Inter-Rater Agreement (Cohen's Kappa)", fontsize=9, pad=10)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0, labelsize=8)
ax.figure.axes[-1].yaxis.label.set_size(8)

# Ensure the plot is displayed cleanly
plt.tight_layout()

plt.show()

In [ ]:
fig.savefig(FIGURES_DIR / "q2-heatmap_cohen_kappa.pdf", dpi=300, bbox_inches='tight')

In [ ]:
all_q2_answers["P3"].eq(all_q2_answers["Phi-2"]).sum()

In [ ]:
### Fleiss' Kappa
from statsmodels.stats.inter_rater import aggregate_raters, fleiss_kappa

fleiss_values = dict()
# Human-Human Fleiss' Kappa
human_raters = q2_df.columns.tolist()
data, n_cat = aggregate_raters(all_q2_answers[human_raters])
fleiss_values["Human"] = fleiss_kappa(data, method='fleiss')

data, n_cat = aggregate_raters(all_q2_answers[human_raters + ["Phi-2"]])
fleiss_values["Human + Phi-2"] = fleiss_kappa(data, method='fleiss')

data, n_cat = aggregate_raters(all_q2_answers[human_raters + ["Gemini"]])
fleiss_values["Human + Gemini"] = fleiss_kappa(data, method='fleiss')

data, n_cat = aggregate_raters(all_q2_answers[human_raters + ["Phi-2", "Gemini"]])
fleiss_values["Human + Both"] = fleiss_kappa(data, method='fleiss')

fleiss_values

### Confusion Matrix

In [ ]:
human_majority = q2_df.mode(axis=1).dropna(axis=1).rename(columns={0: "Human Majority"}).squeeze()
human_majority = human_majority.reindex_like(all_q2_answers)
human_majority

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import seaborn as sns

import matplotlib.pyplot as plt

# Set professional plot style and font
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Linux Libertine"],
})
plt.style.use('seaborn-v0_8-whitegrid')

# Define true and predicted labels
y_true = all_q2_answers["Gemini"] 
y_pred = all_q2_answers["Phi-2"]

# Create the plot
fig, ax = plt.subplots(figsize=(3, 2.5), dpi=300)
ax.grid(False)  # Turn off grid lines for a cleaner heatmap look

# Generate and plot the confusion matrix
disp = ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    ax=ax,
    cmap='viridis',
    colorbar=False,  # Cleaner for a 2x2 matrix
)

# Remove the border around the main heatmap
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)

# Customize plot labels and title for publication
# ax.set_title("Human Majority vs. Phi-2", fontsize=9, pad=10)
ax.set_xlabel("Phi-2", fontsize=8)
ax.set_ylabel("Gemini", fontsize=8)
ax.tick_params(axis='both', which='major', labelsize=8)

# Ensure the plot is displayed cleanly
plt.tight_layout()
plt.show()

In [ ]:
fig.savefig(FIGURES_DIR / "q2-conf_matrix_gemini_phi2.pdf", dpi=300, bbox_inches='tight')

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(human_majority, phi2_q2_answers, zero_division=0))

In [ ]:
print(classification_report(human_majority, gemini_q2_answers, zero_division=0))

In [ ]:
from sklearn.metrics import roc_auc_score

roc_auc_score(human_majority.map({"Yes": 1, "No": 0}), all_q2_answers["Phi-2"].map({"Yes": 1, "No": 0}))

In [ ]:
roc_auc_score(human_majority.map({"Yes": 1, "No": 0}), all_q2_answers["Gemini"].map({"Yes": 1, "No": 0}))

### Overal accuracy

In [ ]:
aligned_df = pd.DataFrame({q_id: [val] * q2_df.shape[1] for q_id, val in aligned.items()}).T
aligned_df

In [ ]:
q2_df.eq(aligned_df).mean(axis=None)

In [ ]:
for i in range(q2_df.shape[1]):
    gemini_q2_df[i] = gemini_q2_df["gemini_q2"]
gemini_q2_df = gemini_q2_df.drop(columns=["gemini_q2"])
gemini_q2_df.index.name = None
gemini_q2_df

In [ ]:
q2_df.eq(gemini_q2_df).mean(axis=None)

### Majority voting

In [ ]:
q2_df.mode(axis=1).dropna(axis=1).eq(aligned_df.iloc[:, [0]]).mean(axis=None)

In [ ]:
q2_df.mode(axis=1).dropna(axis=1).eq(gemini_q2_df.iloc[:, [0]]).mean(axis=None)

In [ ]:
aligned_df.iloc[:, [0]].eq(gemini_q2_df.iloc[:, [0]]).mean(axis=None)

In [ ]:
gemini_q2_df.iloc[:, [0]]

### Confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

y_true = aligned_df.iloc[:, [0]]
y_pred = q2_df.mode(axis=1).dropna(axis=1)

ConfusionMatrixDisplay.from_predictions(y_true, y_pred)
None

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred))

In [ ]:
y_pred = q2_df.to_numpy().reshape(-1)
y_true = aligned_df.to_numpy().reshape(-1)
ConfusionMatrixDisplay.from_predictions(y_true, y_pred)
None

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred))